# Lab L2.1 · Automated valuation model (AVM) for mortgage collateral
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 2 · ⏱ 85 min · **Level 2 · Guided starter (fill in the blanks) · ★ recommended level**

Build, regularise and stress-test a linear-regression AVM on 200k UAE property valuations — and prove it beats the valuer’s rule of thumb.

### How this notebook works
- Every `✏️ BLANK n` marks code **you** write. Inline blanks look like `____`; block blanks say `# ...your code here...`.
- The markdown cell above each code cell explains **what** to write, **where**, and **why**.
- Stuck > 5 min? Open the 💡 hint in the HTML lab guide, then ask your trainer. The Level 3 solution is released after the lab.

### Scenario
**Horizon Gulf Bank (HGB)** is a fictional UAE retail bank with 250,000 customers. Its Board approved **Project MIZAN** — an AI decisioning platform covering credit, fraud, collateral valuation, cash logistics, customer retention and document automation. You are a junior ML practitioner in the MIZAN squad.

The mortgage desk wants an AVM to (a) pre-screen top-up requests and (b) revalue the whole collateral book every quarter for IFRS 9 LTV monitoring. Model Risk will only approve a model that is **interpretable** (coefficients), **accurate** (median error within ±10%), and **stable over time**. A linear model is therefore the right first choice.

**Progressive architecture:** reuses the `ColumnTransformer` pattern you built in Lab L1.2; the AVM is served alongside other MIZAN models on Day 5.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Load, sanity-check and remove data-entry errors

**Why this matters.** Valuation feeds contain keying errors (an extra zero, a missing zero). A ×10 error on a AED 2M villa is a AED 18M squared-error disaster that drags every coefficient. We detect them with a **domain-relative** rule: price-per-sqft compared with the community median.

#### ✏️ What to write in the next cell
- **Blank 1** *(inline)* — compare like with like
- **Blank 2** *(inline)* — a 4× deviation from the local median is not a real price

In [ ]:
raw = pd.read_csv(DATA / "property_valuations.csv", parse_dates=["valuation_date"])
raw["psf"] = raw.valuation_aed / raw.size_sqft
raw["psf_ratio"] = raw.psf / raw.groupby(____).psf.transform("median")  # ✏️ B1
raw["entry_error"] = ~raw.psf_ratio.between(____)  # ✏️ B2
print("entry errors:", raw.entry_error.sum(), f"({raw.entry_error.mean():.2%})")
display(raw.loc[raw.entry_error, ["community", "property_type", "size_sqft", "valuation_aed", "psf_ratio"]].head())
val = raw[~raw.entry_error].copy()
val.describe()[["size_sqft", "valuation_aed", "psf"]].round(0)

**✔ Checkpoint.** ≈ 300 entry errors (0.15%) with ratios ≈ 0.1 or ≈ 10 — the classic extra/missing zero.

## Step 2 · Split and build strong baselines

**Why this matters.** We hold out 15% as a **test set** that is touched once at the end. Then we measure two baselines on validation: the median and the **valuer rule** (community psf × size). The rule is computed from *training* rows only — otherwise it leaks.

#### ✏️ What to write in the next cell
- **Blank 3** *(inline)* — root of the mean squared error — same unit as AED
- **Blank 4** *(inline)* — share of valuations within ±10% of the actual

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
val["months_since_2018"] = (val.valuation_date - pd.Timestamp("2018-01-01")).dt.days / 30.44
train, temp = train_test_split(val, test_size=0.30, random_state=RNG)
valid, test = train_test_split(temp, test_size=0.50, random_state=RNG)
print(len(train), len(valid), len(test))

def reg_metrics(y, p) -> dict:
    y, p = np.asarray(y), np.asarray(p)
    return {"MAE": mean_absolute_error(y, p), "RMSE": ____,  # ✏️ B3
            "R2": r2_score(y, p), "MAPE": mean_absolute_percentage_error(y, p),
            "within_10pct": ____}  # ✏️ B4

results = {}
results["baseline: median"] = reg_metrics(valid.valuation_aed, np.full(len(valid), train.valuation_aed.median()))
psf_med = train.groupby(["community", "property_type"]).psf.median()
rule = valid.set_index(["community", "property_type"]).index.map(psf_med).to_numpy() * valid.size_sqft.to_numpy()
results["baseline: valuer rule"] = reg_metrics(valid.valuation_aed, rule)
pd.DataFrame(results).T.style.format({"MAE": "{:,.0f}", "RMSE": "{:,.0f}", "R2": "{:.3f}", "MAPE": "{:.1%}", "within_10pct": "{:.1%}"})

**✔ Checkpoint.** Median baseline R² ≈ 0 (MAPE ≈ 90%). Valuer rule: R² ≈ 0.89, MAPE ≈ 18%, only ≈ 33% within ±10%. This is the bar to beat.

## Step 3 · Simple linear regression — read the coefficients

**Why this matters.** With one feature the model is a line: **price = intercept + slope × size**. Reading the slope in AED per sqft and plotting residuals against fitted values reveals the two problems with modelling raw prices: **heteroskedasticity** and negative predictions.

#### ✏️ What to write in the next cell
- **Blank 5** *(inline)* — residual = actual − predicted

In [ ]:
from sklearn.linear_model import LinearRegression
slr = LinearRegression().fit(train[["size_sqft"]], train.valuation_aed)
print(f"price ≈ {slr.intercept_:,.0f} + {slr.coef_[0]:,.0f} × size_sqft")
p = slr.predict(valid[["size_sqft"]])
res = ____  # ✏️ B5
smp = np.random.default_rng(RNG).choice(len(valid), 5000, replace=False)
plt.scatter(p[smp], res.iloc[smp], s=4, alpha=.4); plt.axhline(0, color="k")
plt.xlabel("fitted value (AED)"); plt.ylabel("residual (AED)"); plt.title("Funnel shape = heteroskedastic errors"); plt.show()
print("negative predictions:", (p < 0).sum(), "| R² =", round(r2_score(valid.valuation_aed, p), 3))

**✔ Checkpoint.** Slope ≈ AED 700–1,000 per sqft; residuals fan out as price grows; R² well below the valuer rule.

## Step 4 · Multiple linear regression with a log target

**Why this matters.** Property prices are **multiplicative**: a sea view adds a *percentage*, not a fixed AED amount. Modelling **log(price)** with log(size) makes the relationship linear, stabilises the variance and makes coefficients read as **% effects**. `TransformedTargetRegressor` applies `log` for training and `exp` for predictions automatically.

#### ✏️ What to write in the next cell
- **Blank 6** *(inline)* — flexible curve for the 2018-2025 market cycle
- **Blank 7** *(inline)* — train on log(price), predict in AED

In [ ]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, SplineTransformer

NUM = ["bedrooms", "building_age_years", "floor", "distance_to_metro_km", "service_charge_aed_sqft", "parking_spaces"]
CAT = ["community", "property_type", "view", "emirate"]
BIN = ["furnished", "private_pool", "off_plan"]
prep = ColumnTransformer([
    ("logsize", Pipeline([("log", FunctionTransformer(np.log, feature_names_out="one-to-one")), ("sc", StandardScaler())]), ["size_sqft"]),
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
    ("time", ____, ["months_since_2018"]),  # ✏️ B6
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT),
    ("bin", "passthrough", BIN)], verbose_feature_names_out=False)

raw_lr = Pipeline([("prep", prep), ("lr", LinearRegression())]).fit(train, train.valuation_aed)
log_lr = ____  # ✏️ B7
log_lr.fit(train, train.valuation_aed)
results["linear: raw target"] = reg_metrics(valid.valuation_aed, raw_lr.predict(valid))
results["linear: log target"] = reg_metrics(valid.valuation_aed, log_lr.predict(valid))
display(pd.DataFrame(results).T.style.format({"MAE": "{:,.0f}", "RMSE": "{:,.0f}", "R2": "{:.3f}", "MAPE": "{:.1%}", "within_10pct": "{:.1%}"}))

coefs = pd.Series(log_lr.regressor_.named_steps["lr"].coef_, index=log_lr.regressor_.named_steps["prep"].get_feature_names_out())
print((np.exp(coefs.filter(like="view_")) - 1).sort_values().map("{:+.1%}".format))

**✔ Checkpoint.** Log-target model: R² ≈ 0.97+, MAPE ≈ 9%, ≈ 60%+ within ±10% — clearly beats the valuer rule. Sea/Marina view ≈ +20–30% vs Road view.

## Step 5 · Regularisation: Ridge vs Lasso with cross-validated alpha

**Why this matters.** With ~60 correlated features, coefficients can become unstable. **Ridge (L2)** shrinks them smoothly; **Lasso (L1)** can set them to **exactly zero** — automatic feature selection. The strength `alpha` is a hyperparameter, chosen by 5-fold CV on the *training* set.

#### ✏️ What to write in the next cell
- **Blank 8** *(inline)* — 13 values from 0.001 to 1000

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import GridSearchCV, KFold
cv = KFold(5, shuffle=True, random_state=RNG)
def log_model(est):
    return TransformedTargetRegressor(regressor=Pipeline([("prep", prep), ("m", est)]), func=np.log, inverse_func=np.exp)

ridge_gs = GridSearchCV(log_model(Ridge()), {"regressor__m__alpha": ____},  # ✏️ B8
                        cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1).fit(train, train.valuation_aed)
lasso_gs = GridSearchCV(log_model(Lasso(max_iter=5000)), {"regressor__m__alpha": np.logspace(-5, -1, 5)},
                        cv=3, scoring="neg_mean_absolute_error", n_jobs=-1).fit(train, train.valuation_aed)
for name, gs in [("ridge", ridge_gs), ("lasso", lasso_gs)]:
    print(name, "best alpha:", gs.best_params_)
    results[f"{name} (tuned)"] = reg_metrics(valid.valuation_aed, gs.predict(valid))
display(pd.DataFrame(results).T.style.format({"MAE": "{:,.0f}", "RMSE": "{:,.0f}", "R2": "{:.3f}", "MAPE": "{:.1%}", "within_10pct": "{:.1%}"}))

In [ ]:
Xp = prep.fit_transform(train); yp = np.log(train.valuation_aed)
alphas = np.logspace(-5, -0.5, 15)
nonzero = [(Lasso(alpha=a, max_iter=5000).fit(Xp, yp).coef_ != 0).sum() for a in alphas]
plt.semilogx(alphas, nonzero, "o-"); plt.xlabel("Lasso alpha"); plt.ylabel("# non-zero coefficients")
plt.title(f"Lasso feature selection (of {Xp.shape[1]} features)"); plt.show()

**✔ Checkpoint.** CV picks the *smallest* alphas and Ridge/Lasso tie with plain OLS (≈ 9.2% MAPE): with 140k rows and ~60 features there is little variance to remove. The Lasso path shows features dropping to zero as alpha grows.

## Step 6 · Final test evaluation and segment error analysis

**Why this matters.** Overall metrics hide pockets of failure. A bank cares whether the AVM is reliable *for villas in Palm Jumeirah*, not just on average. We evaluate on the untouched **test** set once, then break the error down by segment and inspect the worst cases.

#### ✏️ What to write in the next cell
- **Blank 9** *(inline)* — segments the mortgage desk manages

In [ ]:
best_name = min([k for k in results if not k.startswith("baseline")], key=lambda k: results[k]["MAE"])
final = {"linear: log target": log_lr, "ridge (tuned)": ridge_gs.best_estimator_, "lasso (tuned)": lasso_gs.best_estimator_}.get(best_name, ridge_gs.best_estimator_)
print("selected:", best_name)
test = test.assign(pred=final.predict(test))
test["ape"] = np.abs(test.pred / test.valuation_aed - 1)
print({k: round(v, 4) for k, v in reg_metrics(test.valuation_aed, test.pred).items()})
seg = test.groupby(____).agg(n=("ape", "size"), MAPE=("ape", "mean"), within_10=("ape", lambda s: (s <= .1).mean()))  # ✏️ B9
display(seg.sort_values("MAPE", ascending=False).round(3))
plt.figure(figsize=(5, 5)); plt.loglog(test.valuation_aed, test.pred, ".", ms=1, alpha=.3)
lim = [test.valuation_aed.min(), test.valuation_aed.max()]; plt.plot(lim, lim, "r-")
plt.xlabel("actual"); plt.ylabel("predicted"); plt.title("Test: predicted vs actual"); plt.show()
test.nlargest(5, "ape")[["community", "property_type", "size_sqft", "view", "valuation_aed", "pred", "ape"]]

**✔ Checkpoint.** Test MAPE ≈ 9%, R² ≈ 0.97. Errors are fairly even across segments; the worst cases are unusual properties (very large, very high floor or old).

## Step 7 · Out-of-time stress test and model hand-off

**Why this matters.** Random splits let the model see every month of the market. In production it values **tomorrow’s** properties. Training on 2018–2024 and testing on 2025 shows how fast the AVM ages — the input to its **recalibration policy**.

#### ✏️ What to write in the next cell
- **Blank 10** *(inline)* — positive = over-valuation

In [ ]:
import joblib, json
from sklearn.base import clone
past, future = val[val.valuation_date < "2025-01-01"], val[val.valuation_date >= "2025-01-01"].copy()
oot = clone(final).fit(past, past.valuation_aed)
future["pred"] = oot.predict(future)
future["signed_err"] = ____  # ✏️ B10
q = future.groupby(future.valuation_date.dt.to_period("Q")).agg(MAPE=("signed_err", lambda s: s.abs().mean()), median_bias=("signed_err", "median"), n=("pred", "size"))
display(q.round(4))
metrics = {"model": best_name, "test": {k: float(v) for k, v in reg_metrics(test.valuation_aed, test.pred).items()},
           "oot_2025_mape": float(future.signed_err.abs().mean()), "oot_2025_median_bias": float(future.signed_err.median()),
           "trained_rows": int(len(train)), "outlier_rule": "psf ratio to community×type median outside [0.25, 4]"}
joblib.dump(final, CKPT / "valuation_model.joblib")
(CKPT / "valuation_metrics.json").write_text(json.dumps(metrics, indent=2)); print(json.dumps(metrics, indent=2))

**✔ Checkpoint.** OOT MAPE is a little worse than the random-split test, with a systematic bias (the spline cannot extrapolate the 2025 market) — evidence for **quarterly recalibration**.

## ✅ Lab complete
The AVM beats the valuer rule by a wide margin and is saved with its metrics. **Next:** forecasting — where time order changes everything.